# 简单的 RAG（检索增强生成）系统

## 概述

此代码实现了用于处理和查询 PDF 文档的基本检索增强生成 (RAG) 系统。系统将文档内容编码到向量存储中，然后可以查询该向量存储以检索相关信息。

## 关键组件

1. PDF 处理和文本提取
2. 文本分块以实现易于管理的处理
3. 使用 [FAISS](https://engineering.fb.com/2017/03/29/data-infrastructure/faiss-a-library-for-efficient-similarity-search/) 和 OpenAI 嵌入创建向量存储
4. 用于查询已处理文档的检索器设置
5. RAG 系统的评估

## 方法详细信息

### 文档预处理

1. 使用 PyPDFLoader加载PDF。
2. 使用 RecursiveCharacterTextSplitter 将文本拆分为具有指定块大小和重叠的块。

### 文本清理

应用自定义函数 `replace_t_with_space` 来清理文本块。这可能会解决 PDF 中的特定格式问题。

### 向量存储创建

1. OpenAI 嵌入用于创建文本块的向量表示。
2. 根据这些嵌入创建 FAISS 向量存储，以进行高效的相似性搜索。

### 检索器设置

1. 检索器配置为获取给定查询的前 2 个最相关的块。

### 编码函数

`encode_pdf` 函数封装了将 PDF 加载、分块、清理和编码到向量存储的整个过程。

## 主要特点

1. 模块化设计：编码过程封装在单个函数中，方便重用。
2. 可配置分块：允许调整块大小和重叠。
3. 高效检索：使用FAISS进行快速相似性搜索。
4. 评估：包括评估RAG 系统性能的功能。

## 使用示例

该代码包含一个测试查询：“气候变化的主要原因是什么？”。这演示了如何使用检索器从处理后的文档中获取相关上下文。

## 评价

该系统包括一个 `evaluate_rag` 函数来评估检索器的性能，尽管所提供的代码中没有详细说明所使用的具体指标。

## 这种方法的好处

1. 可扩展性：可以通过分块处理大型文档。
2. 灵活性：易于调整块大小和检索结果数量等参数。
3. 高效：利用FAISS在高维空间中进行快速相似性搜索。
4. 与高级 NLP 集成：使用 OpenAI 嵌入来实现最先进的文本表示。

## 结论

这个简单的 RAG 系统为构建更复杂的信息检索和问答系统提供了坚实的基础。通过将文档内容编码到可搜索的向量存储中，它可以响应查询而有效地检索相关信息。此方法对于需要快速访问大型文档或文档集合中的特定信息的应用程序特别有用。

# 包安装和导入

下面的单元格安装运行此笔记本所需的所有必需软件包。

In [2]:
# Install required packages
!pip install pypdf==5.6.0
!pip install PyMuPDF==1.26.1
!pip install python-dotenv==1.1.0
!pip install langchain-community==0.3.25
!pip install langchain_openai==0.3.23
!pip install rank_bm25==0.2.2
!pip install faiss-cpu==1.11.0
!pip install deepeval==3.1.0

In [8]:
# Clone the repository to access helper functions and evaluation modules
# !git clone https://github.com/NirDiamant/RAG_TECHNIQUES.git
import sys
sys.path.append('/home/test/Desktop/code/RAG_Techniques')

# If you need to run with the latest data
# !cp -r RAG_TECHNIQUES/data .

In [14]:
import os
print(os.getcwd())

/home/test/Desktop/code/RAG_Techniques/all_rag_techniques


In [ ]:
import os
import sys
from dotenv import load_dotenv
# from google.colab import userdata



# Load environment variables from a .env file
load_dotenv()

# Set the OpenAI API key environment variable (comment out if not using OpenAI)
# if not userdata.get('OPENAI_API_KEY'):
#     os.environ["OPENAI_API_KEY"] = input("Please enter your OpenAI API key: ")
# else:
#     os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Original path append replaced for Colab compatibility

from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from helper_functions import (EmbeddingProvider,
                              retrieve_context_per_question,
                              replace_t_with_space,
                              get_langchain_embedding_provider, 
                              show_context)

# from evaluation.evalute_rag import evaluate_rag

from langchain.vectorstores import FAISS


### 阅读文档

In [12]:
# Download required data files
import os
os.makedirs('data', exist_ok=True)

# Download the PDF document used in this notebook
!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf
# !wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf


--2026-04-09 15:10:39--  https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 206372 (202K) [application/octet-stream]
Saving to: ‘data/Understanding_Climate_Change.pdf’

data/Understanding_ 100%[===================>] 201.54K  --.-KB/s    in 0.09s   

2026-04-09 15:10:40 (2.16 MB/s) - ‘data/Understanding_Climate_Change.pdf’ saved [206372/206372]



In [15]:
path = "data/Understanding_Climate_Change.pdf"

### 编码文档

In [18]:
def encode_pdf(path, chunk_size=1000, chunk_overlap=200):
    """
    Encodes a PDF book into a vector store using OpenAI embeddings.

    Args:
        path: The path to the PDF file.
        chunk_size: The desired size of each text chunk.
        chunk_overlap: The amount of overlap between consecutive chunks.

    Returns:
        A FAISS vector store containing the encoded book content.
    """

    # Load PDF documents
    loader = PyPDFLoader(path)
    documents = loader.load()
    print(documents)

    # Split documents into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len
    )
    texts = text_splitter.split_documents(documents)
    cleaned_texts = replace_t_with_space(texts)

    # Create embeddings (Tested with OpenAI and Amazon Bedrock)
    embeddings = get_langchain_embedding_provider(EmbeddingProvider.OPENAI)
    #embeddings = get_langchain_embedding_provider(EmbeddingProvider.AMAZON_BEDROCK)

    # Create vector store
    vectorstore = FAISS.from_documents(cleaned_texts, embeddings)

    return vectorstore

In [19]:
chunks_vector_store = encode_pdf(path, chunk_size=1000, chunk_overlap=200)

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2024-07-13T20:17:34+03:00', 'author': 'Nir', 'moddate': '2024-07-13T20:17:34+03:00', 'source': 'data/Understanding_Climate_Change.pdf', 'total_pages': 33, 'page': 0, 'page_label': '1'}, page_content='Understanding Climate Change \nChapter 1: Introduction to Climate Change \nClimate change refers to significant, long-term changes in the global climate. The term \n"global climate" encompasses the planet\'s overall weather patterns, including temperature, \nprecipitation, and wind patterns, over an extended period. Over the past century, human \nactivities, particularly the burning of fossil fuels and deforestation, have significantly \ncontributed to climate change. \nHistorical Context \nThe Earth\'s climate has changed throughout history. Over the past 650,000 years, there have \nbeen seven cycles of glacial advance and retreat, with the abrupt end of the last ice age about \n11,

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

### 创建搜索器

In [7]:
chunks_query_retriever = chunks_vector_store.as_retriever(search_kwargs={"k": 2})

### 测试检索器

In [8]:
test_query = "What is the main cause of climate change?"
context = retrieve_context_per_question(test_query, chunks_query_retriever)
show_context(context)

Context 1:
Chapter 2: Causes of Climate Change 
Greenhouse Gases 
The primary cause of recent climate change is the increase in greenhouse gases in the 
atmosphere. Greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous 
oxide (N2O), trap heat from the sun, creating a "greenhouse effect." This effect is essential 
for life on Earth, as it keeps the planet warm enough to support life. However, human 
activities have intensified this natural process, leading to a warmer climate. 
Fossil Fuels 
Burning fossil fuels for energy releases large amounts of CO2. This includes coal, oil, and 
natural gas used for electricity, heating, and transportation. The industrial revolution marked 
the beginning of a significant increase in fossil fuel consumption, which continues to rise 
today. 
Coal


Context 2:
Most of these climate changes are attributed to very small variations in Earth's orbit that 
change the amount of solar energy our planet receives. During the Holocene epoch,

/content/RAG_TECHNIQUES/helper_functions.py:143: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = chunks_query_retriever.get_relevant_documents(question)


### 评估结果

In [9]:
#Note - this currently works with OPENAI only
evaluate_rag(chunks_query_retriever)

{'questions': ['1. **Multiple Choice: Causes of Climate Change**',
  '   - What is the primary cause of the current climate change trend?',
  '     A) Solar radiation variations',
  '     B) Natural cycles of the Earth',
  '     C) Human activities, such as burning fossil fuels',
  '     D) Volcanic eruptions',
  '',
  '2. **True or False: Climate Change Impacts**',
  '   - True or False: Climate change only affects the temperature of the planet, not weather patterns, sea levels, or ecosystems.',
  '',
  '3. **Short Answer: Mitigation Strategies**',
  '   - Describe two effective strategies that could be implemented to mitigate the effects of climate change.',
  '',
  '4. **Matching: Climate Change Terminology**',
  '   - Match the following terms with their correct definitions:',
  '     A) Greenhouse Gases',
  '     B) Carbon Footprint',
  '     C) Renewable Energy',
  '     D) Adaptation',
  '     - Definitions:',
  '       1. The total amount of greenhouse gases produced to directl

![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--simple-rag)